# Bloc 13 — Sustainable Machine Learning
## Performance prédictive, coût de calcul et sobriété du pipeline

### Partie A — Mise en pratique

## Étape 1 — Métriques de Sustainable ML

L'objectif de ce bloc est d'étudier le compromis entre performance
prédictive et coût informatique.

Le projet ne cherche pas seulement à minimiser l'erreur de prédiction.

Il cherche également à éviter une complexité inutile lorsque le gain de
performance devient très faible.

Les indicateurs étudiés sont :

- RMSE ;
- MAE ;
- temps d'inférence ;
- taille du modèle sur disque ;
- complexité du pipeline ;
- gain de performance par rapport à un modèle simple.

L'année 2026 reste totalement exclue de cette analyse.

In [1]:
# ==========================================
# BLOCK 13 — INITIALISATION
# ==========================================

from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import time
import os


PROJECT_DIR = Path(
    r"C:\Users\thiba\OneDrive\Bureau\Projet energie"
)

MODEL_DIR = (
    PROJECT_DIR
    / "models"
)

DATA_DIR = (
    PROJECT_DIR
    / "data_processed"
)


BLOCK13_DIR = (
    DATA_DIR
    / "block13"
)


BLOCK13_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


RANDOM_STATE = 42


print(
    "Block 13 initialisé."
)

Block 13 initialisé.


In [2]:
# ==========================================
# SUSTAINABLE ML CONFIGURATION
# ==========================================

RETRAIN_MODELS = False

USE_2026 = False

MEASURE_MODEL_SIZE = True

MEASURE_INFERENCE_TIME = True


block13_config = {

    "retrain_models":
        RETRAIN_MODELS,

    "use_2026":
        USE_2026,

    "measure_model_size":
        MEASURE_MODEL_SIZE,

    "measure_inference_time":
        MEASURE_INFERENCE_TIME,
}


display(
    pd.Series(
        block13_config
    )
)

retrain_models            False
use_2026                  False
measure_model_size         True
measure_inference_time     True
dtype: bool

In [3]:
# ==========================================
# LOAD BLOCK 7 STATE
# ==========================================

BLOCK7_PATH = (
    MODEL_DIR
    / "block7_final_state.joblib"
)


assert (
    BLOCK7_PATH.exists()
)


block7_state = joblib.load(
    BLOCK7_PATH
)


print(
    "Bloc 7 chargé."
)

Bloc 7 chargé.


In [4]:
# ==========================================
# BLOCK 7 CONTENT
# ==========================================

if isinstance(
    block7_state,
    dict
):

    print(
        "Clés disponibles :"
    )

    for key in block7_state.keys():

        print(
            "-",
            key
        )

else:

    print(
        "Type :",
        type(
            block7_state
        )
    )

Clés disponibles :
- model_name
- index_2025
- y_2025
- prediction_2025
- metrics_2025
- rmse_2025
- event_columns
- event_lambda
- ewma_selected
- ewma_alpha
- dst_expert_selected
- dm_summary


In [5]:
# ==========================================
# FINAL STATE SIZE
# ==========================================

block7_size_bytes = (
    BLOCK7_PATH.stat().st_size
)


block7_size_mb = (
    block7_size_bytes
    / 1024**2
)


print(
    "Taille Block 7 state :",
    f"{block7_size_mb:.3f} MB"
)

Taille Block 7 state : 0.404 MB


Le fichier d'état final du Bloc 7 occupe environ 0.404 MB.

Cette taille ne représente pas nécessairement la taille totale du système
d'inférence : l'état peut contenir des métadonnées ou des références vers
des composants enregistrés séparément.

La taille complète du pipeline sera donc vérifiée avant toute conclusion
sur son coût de stockage.

In [6]:
# ==========================================
# BLOCK 7 REFERENCE PERFORMANCE
# ==========================================

FINAL_RMSE = (
    176.942561
)

FINAL_MAE = (
    127.851620
)

FINAL_MAPE = (
    0.2576117
)

FINAL_R2 = (
    0.999722377
)


reference_performance = pd.DataFrame({

    "Metric": [
        "RMSE",
        "MAE",
        "MAPE (%)",
        "R2",
    ],

    "Value": [
        FINAL_RMSE,
        FINAL_MAE,
        FINAL_MAPE,
        FINAL_R2,
    ],
})


display(
    reference_performance
)

,Metric,Value
0,RMSE,176.942561
1,MAE,127.851620
2,MAPE (%),0.257612
3,R2,0.999722


In [7]:
RIDGE_RMSE = (
    209.6614
)


rmse_improvement = (
    RIDGE_RMSE
    - FINAL_RMSE
)


rmse_improvement_pct = (
    rmse_improvement
    / RIDGE_RMSE
    * 100
)


print(
    "Gain RMSE absolu :",
    f"{rmse_improvement:.3f} MW"
)

print(
    "Gain RMSE relatif :",
    f"{rmse_improvement_pct:.2f}%"
)

Gain RMSE absolu : 32.719 MW
Gain RMSE relatif : 15.61%


### Mesurer l'efficacité d'un modèle

Un modèle plus complexe n'est intéressant que si l'amélioration obtenue
justifie le coût supplémentaire.

On peut représenter le problème par deux axes :

$$
\text{Qualité prédictive}
$$

et :

$$
\text{Coût informatique}.
$$

Par exemple, si deux modèles possèdent presque le même RMSE mais que le
second nécessite dix fois plus de calcul, le modèle plus simple peut être
préférable dans certaines situations opérationnelles.

Inversement, une augmentation du coût peut être justifiée si elle produit
une amélioration prédictive suffisamment importante.

Le Sustainable ML est donc étudié ici comme un problème de compromis et
non comme la recherche systématique du modèle le moins coûteux.

In [8]:
# ==========================================
# SUSTAINABILITY METRICS
# ==========================================

sustainability_metrics = pd.DataFrame({

    "Metric": [

        "RMSE",

        "MAE",

        "Model size",

        "Inference time",

        "Pipeline complexity",

    ],

    "Interpretation": [

        "Erreur prédictive",

        "Erreur absolue moyenne",

        "Mémoire / stockage nécessaire",

        "Coût d'utilisation du modèle",

        "Nombre de composants nécessaires",

    ],
})


display(
    sustainability_metrics
)

,Metric,Interpretation
0,RMSE,Erreur prédictive
1,MAE,Erreur absolue moyenne
2,Model size,Mémoire / stockage nécessaire
3,Inference time,Coût d'utilisation du modèle
4,Pipeline complexity,Nombre de composants nécessaires


### Limite de l'estimation carbone

Le temps de calcul ne peut pas être converti directement en émissions de
CO₂ avec suffisamment de précision sans connaître notamment :

- la consommation électrique réelle du matériel ;
- l'utilisation du CPU et du GPU ;
- la durée exacte des calculs ;
- le mix électrique utilisé pendant l'exécution.

Le projet privilégie donc des métriques directement observables :

- temps d'exécution ;
- taille des modèles ;
- nombre de composants ;
- performance prédictive.

Une estimation carbone pourra être ajoutée ultérieurement si des mesures
matérielles suffisamment fiables sont disponibles.

In [9]:
# ==========================================
# SAVE STEP 1
# ==========================================

STEP1_PATH = (
    BLOCK13_DIR
    / "step1_sustainable_config.joblib"
)


step1_state = {

    "configuration":
        block13_config,

    "block7_size_mb":
        block7_size_mb,

    "final_rmse":
        FINAL_RMSE,

    "ridge_rmse":
        RIDGE_RMSE,

    "rmse_improvement_pct":
        rmse_improvement_pct,

    "uses_2026":
        False,
}


joblib.dump(
    step1_state,
    STEP1_PATH
)


print(
    "Sauvegardé :",
    STEP1_PATH
)

Sauvegardé : C:\Users\thiba\OneDrive\Bureau\Projet energie\data_processed\block13\step1_sustainable_config.joblib


In [10]:
# ==========================================
# STEP 1 VALIDATION
# ==========================================

assert (
    RETRAIN_MODELS
    is False
)

assert (
    USE_2026
    is False
)

assert (
    FINAL_RMSE
    < RIDGE_RMSE
)

assert (
    block7_size_mb
    > 0
)


print(
    "Étape 1 : OK"
)

Étape 1 : OK


### Partie C — Analyse et améliorations guidées par la théorie

La performance prédictive ne constitue pas le seul critère permettant
d'évaluer un système de Machine Learning.

Le modèle final du projet atteint un RMSE d'environ :

$$
176.94\ MW,
$$

contre environ :

$$
209.66\ MW
$$

pour le modèle Ridge de référence.

L'amélioration relative du RMSE est donc d'environ :

$$
15.6\%.
$$

Cette amélioration justifie l'étude du pipeline complexe, mais il reste à
déterminer le coût informatique associé à ce gain.

Le Sustainable ML est donc traité comme un compromis entre :

$$
\text{performance}
\leftrightarrow
\text{coût informatique}.
$$

Le projet évite également de réentraîner inutilement les modèles déjà
calculés afin de limiter le coût expérimental du Bloc 13.

Conclusion :

L'Étape 1 définit les critères utilisés pour analyser la sobriété du
pipeline.

Le modèle final améliore le RMSE d'environ 15.6 % par rapport au Ridge de
référence.

Cette amélioration sera maintenant mise en regard de la taille des modèles,
du temps d'inférence et de la complexité du pipeline.

Les modèles existants sont réutilisés sans nouvel entraînement et aucune
donnée de 2026 n'est utilisée.

## Étape 2 — Audit du pipeline final

### Partie A — Mise en pratique

Le fichier `block7_final_state.joblib` contient les résultats et la
configuration du modèle final, mais pas nécessairement les poids de
tous les modèles utilisés pour produire les prédictions.

L'objectif de cette étape est donc de :

- vérifier les résultats du modèle final ;
- identifier ses différents composants ;
- inventorier les artefacts réellement stockés sur disque ;
- mesurer leur taille lorsque cela est possible.

Aucun modèle n'est réentraîné.

In [13]:
# ==========================================
# FINAL MODEL SUMMARY
# ==========================================

print(
    "Modèle final :",
    block7_state[
        "model_name"
    ]
)

print(
    "\nRMSE 2025 :",
    block7_state[
        "rmse_2025"
    ]
)

print(
    "\nMétriques 2025 :"
)

display(
    pd.Series(
        block7_state[
            "metrics_2025"
        ]
    )
)

Modèle final : Ridge + MLP1 + Patch Transformer + Refined Event Expert

RMSE 2025 : 176.94256118746154

Métriques 2025 :


MSE       31308.669960
RMSE        176.942561
MAE         127.851620
MAPE_%        0.257612
R2            0.999722
dtype: float64

In [14]:
# ==========================================
# PREDICTION CHECK
# ==========================================

y_2025 = (
    np.asarray(
        block7_state[
            "y_2025"
        ]
    )
)

prediction_2025 = (
    np.asarray(
        block7_state[
            "prediction_2025"
        ]
    )
)


print(
    "Observations :",
    len(
        y_2025
    )
)

print(
    "Prédictions :",
    len(
        prediction_2025
    )
)


assert (
    len(
        y_2025
    )
    ==
    len(
        prediction_2025
    )
)


print(
    "Alignement prédictions : OK"
)

Observations : 17517
Prédictions : 17517
Alignement prédictions : OK


In [15]:
# ==========================================
# RMSE VALIDATION
# ==========================================

rmse_check = float(
    np.sqrt(
        np.mean(
            (
                y_2025
                - prediction_2025
            ) ** 2
        )
    )
)


print(
    "RMSE sauvegardé :",
    block7_state[
        "rmse_2025"
    ]
)

print(
    "RMSE recalculé :",
    rmse_check
)


assert np.isclose(
    rmse_check,
    block7_state[
        "rmse_2025"
    ],
)


print(
    "RMSE : OK"
)

RMSE sauvegardé : 176.94256118746154
RMSE recalculé : 176.94256118746154
RMSE : OK


In [16]:
# ==========================================
# FINAL PIPELINE CONFIGURATION
# ==========================================

pipeline_configuration = pd.DataFrame({

    "Element": [
        "Model name",
        "Event correction",
        "EWMA selected",
        "EWMA alpha",
        "DST expert selected",
        "Event lambda",
    ],

    "Value": [
        block7_state[
            "model_name"
        ],

        len(
            block7_state[
                "event_columns"
            ]
        ) > 0,

        block7_state[
            "ewma_selected"
        ],

        block7_state[
            "ewma_alpha"
        ],

        block7_state[
            "dst_expert_selected"
        ],

        block7_state[
            "event_lambda"
        ],
    ],
})


display(
    pipeline_configuration
)

,Element,Value
0,Model name,Ridge + MLP1 + Patch Transformer + Refined Eve...
1,Event correction,True
2,EWMA selected,False
3,EWMA alpha,0.001
4,DST expert selected,False
5,Event lambda,0.5


In [17]:
# ==========================================
# DIEBOLD-MARIANO SUMMARY
# ==========================================

dm_summary = (
    block7_state[
        "dm_summary"
    ]
)


display(
    dm_summary
)

,Test,Model_A,Model_B,DM_stat,p_value,mean_loss_diff
0,MSE — HAC 1 day,Patch,Event Expert,3.197891,0.001384,226.858971
1,MSE — HAC 1 day,Event Expert,Event + EWMA,0.201131,0.840596,3.219023
2,MSE — HAC 1 day,Patch,Event + EWMA,3.173882,0.001504,230.077993
3,MSE — HAC 1 week,Patch,Event Expert,3.309755,0.000934,226.858971
4,MSE — HAC 1 week,Event Expert,Event + EWMA,0.184398,0.853701,3.219023
5,MSE — HAC 1 week,Patch,Event + EWMA,3.286241,0.001015,230.077993
6,MAE — HAC 1 week,Patch,Event Expert,3.414723,0.000638,0.393709
7,MAE — HAC 1 week,Event Expert,Event + EWMA,1.147361,0.251233,0.053366
8,MAE — HAC 1 week,Patch,Event + EWMA,3.616053,0.000299,0.447075


In [18]:
# ==========================================
# MODEL ARTIFACT INVENTORY
# ==========================================

MODEL_EXTENSIONS = {
    ".joblib",
    ".pkl",
    ".pickle",
    ".pt",
    ".pth",
    ".onnx",
    ".keras",
    ".h5",
}


model_files = []


for path in MODEL_DIR.rglob("*"):

    if (
        path.is_file()
        and path.suffix.lower()
        in MODEL_EXTENSIONS
    ):

        model_files.append({

            "file":
                path.name,

            "size_mb":
                (
                    path.stat().st_size
                    / 1024**2
                ),

            "path":
                str(
                    path.relative_to(
                        PROJECT_DIR
                    )
                ),
        })


model_inventory = (
    pd.DataFrame(
        model_files
    )
    .sort_values(
        "size_mb",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)


display(
    model_inventory
)

,file,size_mb,path
0,random_forest_baseline_block2.joblib,47.277268,models\random_forest_baseline_block2.joblib
1,random_forest_tuned_50.joblib,9.777299,models\random_forest_tuned_50.joblib
2,xgboost_block2.joblib,2.380702,models\xgboost_block2.joblib
3,xgb_baseline.joblib,2.310926,models\xgb_baseline.joblib
4,xgboost_depth7_final.joblib,1.900384,models\xgboost_depth7_final.joblib
...,...,...,...
128,block6_gru_residual_scaler.joblib,0.000138,models\block6_gru_residual_scaler.joblib
129,hgb_lr_iter_results.joblib,0.000107,models\hgb_lr_iter_results.joblib
130,block12_config.joblib,0.000081,models\block12_config.joblib
131,xgb_depth_results.joblib,0.000058,models\xgb_depth_results.joblib


In [19]:
# ==========================================
# BLOCK 7 MODEL CANDIDATES
# ==========================================

keywords = [
    "ridge",
    "mlp",
    "patch",
    "transform",
    "event",
    "block7",
]


if not model_inventory.empty:

    pattern = "|".join(
        keywords
    )


    block7_candidates = (
        model_inventory[
            model_inventory[
                "file"
            ]
            .str.lower()
            .str.contains(
                pattern,
                regex=True,
            )
        ]
        .copy()
    )


    display(
        block7_candidates
    )

,file,size_mb,path
8,block7_concept_drift_audit.joblib,1.206416,models\block7_concept_drift_audit.joblib
12,ridge_rank_reduced_357.joblib,1.040841,models\ridge_rank_reduced_357.joblib
13,block7_clean_drift_audit.joblib,0.956008,models\block7_clean_drift_audit.joblib
14,block7_ewma_adaptive_results.joblib,0.670572,models\block7_ewma_adaptive_results.joblib
15,block6_mlp_direct.pt,0.549085,models\block6_mlp_direct.pt
17,block6_ridge_oof_2022.joblib,0.536049,models\block6_ridge_oof_2022.joblib
18,block6_ridge_oof_2023.joblib,0.536049,models\block6_ridge_oof_2023.joblib
21,block7_final_state.joblib,0.403960,models\block7_final_state.joblib
22,block6_residual_patch_transformer.pt,0.292122,models\block6_residual_patch_transformer.pt
23,block7_event_expert_results.joblib,0.286208,models\block7_event_expert_results.joblib


In [20]:
# ==========================================
# CANDIDATE ARTIFACT SIZE
# ==========================================

if not block7_candidates.empty:

    candidate_size_mb = float(
        block7_candidates[
            "size_mb"
        ].sum()
    )


    print(
        "Taille totale des artefacts candidats :",
        f"{candidate_size_mb:.3f} MB"
    )

else:

    candidate_size_mb = np.nan

    print(
        "Aucun artefact identifié automatiquement."
    )

Taille totale des artefacts candidats : 8.624 MB


In [21]:
# ==========================================
# PERFORMANCE GAIN
# ==========================================

comparison_performance = pd.DataFrame({

    "Model": [
        "Ridge",
        "Final composite",
    ],

    "RMSE": [
        RIDGE_RMSE,
        FINAL_RMSE,
    ],
})


comparison_performance[
    "RMSE_gain_vs_Ridge"
] = (
    RIDGE_RMSE
    - comparison_performance[
        "RMSE"
    ]
)


display(
    comparison_performance
)

,Model,RMSE,RMSE_gain_vs_Ridge
0,Ridge,209.661400,0.000000
1,Final composite,176.942561,32.718839


In [22]:
# ==========================================
# PIPELINE COMPLEXITY
# ==========================================

pipeline_components = pd.DataFrame({

    "Component": [
        "Ridge",
        "MLP",
        "Patch Transformer",
        "Refined Event Expert",
    ],

    "Role": [
        "Linear baseline / component",
        "Non-linear component",
        "Temporal deep-learning component",
        "Event correction",
    ],
})


display(
    pipeline_components
)

,Component,Role
0,Ridge,Linear baseline / component
1,MLP,Non-linear component
2,Patch Transformer,Temporal deep-learning component
3,Refined Event Expert,Event correction


### Partie C — Analyse et améliorations guidées par la théorie

Le fichier d'état final du Bloc 7 occupe environ 0.404 MB, mais cette
valeur ne correspond pas à la taille complète du système d'inférence.

Le fichier contient principalement :

- les prédictions 2025 ;
- les observations réelles ;
- les métriques ;
- la configuration du modèle final ;
- les résultats statistiques déjà calculés.

Le pipeline final combine plusieurs composants :

$$
Ridge
+
MLP
+
Patch\ Transformer
+
Event\ Expert.
$$

Cette architecture est plus complexe qu'un simple modèle Ridge, mais
elle réduit le RMSE d'environ 15.6 %.

L'analyse Sustainable ML doit donc distinguer :

$$
\text{taille de l'état sauvegardé}
$$

de :

$$
\text{taille réelle de tous les modèles nécessaires à l'inférence}.
$$

Les métriques et tests déjà disponibles sont réutilisés afin d'éviter
des réentraînements ou calculs inutiles.

Conclusion :

Le modèle final améliore sensiblement la qualité prédictive par rapport
au Ridge, mais il repose sur plusieurs composants supplémentaires.

Le fichier `block7_final_state.joblib` est très léger, avec environ
0.404 MB, mais il contient principalement les résultats du Bloc 7 et ne
représente pas nécessairement la taille complète du système d'inférence.

L'inventaire des artefacts du dossier `models` permet maintenant
d'identifier les fichiers réellement nécessaires et leur coût de
stockage.

Aucun réentraînement n'a été effectué et l'année 2026 reste exclue.

## Étape 3 — Compromis performance / complexité

### Partie A — Mise en pratique

Le modèle final combine plusieurs composants et obtient une meilleure
performance prédictive qu'un Ridge simple.

L'objectif est maintenant de déterminer si cette complexité supplémentaire
apporte un gain suffisamment important pour être conservée.

L'analyse repose sur :

- le gain de RMSE ;
- le nombre de composants ;
- la taille des artefacts ;
- les résultats statistiques déjà obtenus au Bloc 7.

In [23]:
# ==========================================
# PERFORMANCE / COMPLEXITY SUMMARY
# ==========================================

relative_rmse_gain = (
    (RIDGE_RMSE - FINAL_RMSE)
    / RIDGE_RMSE
    * 100
)


step3_summary = pd.DataFrame({

    "Indicator": [
        "Ridge RMSE",
        "Final RMSE",
        "Relative RMSE improvement (%)",
        "Final state size (MB)",
        "Candidate artifacts size (MB)",
        "Number of main components",
    ],

    "Value": [
        RIDGE_RMSE,
        FINAL_RMSE,
        relative_rmse_gain,
        block7_size_mb,
        candidate_size_mb,
        4,
    ],
})


display(
    step3_summary
)

,Indicator,Value
0,Ridge RMSE,209.661400
1,Final RMSE,176.942561
2,Relative RMSE improvement (%),15.605562
3,Final state size (MB),0.403960
4,Candidate artifacts size (MB),8.624491
5,Number of main components,4.000000


### Interprétation de la taille des artefacts

Les artefacts détectés par mots-clés représentent environ 8.6 MB.

Cette valeur ne doit pas être interprétée comme la taille exacte du modèle
final.

L'inventaire contient également des éléments tels que :

- des audits ;
- des résultats intermédiaires ;
- des modèles testés précédemment ;
- des historiques ;
- des configurations.

La valeur de 8.6 MB constitue donc plutôt un indicateur de l'empreinte des
artefacts associés aux Blocs 6 et 7 qu'une mesure exacte de la taille du
pipeline d'inférence final.

In [24]:
# ==========================================
# OPTIONAL COMPONENTS
# ==========================================

optional_components = pd.DataFrame({

    "Component": [
        "EWMA correction",
        "DST expert",
    ],

    "Selected": [
        block7_state[
            "ewma_selected"
        ],

        block7_state[
            "dst_expert_selected"
        ],
    ],
})


display(
    optional_components
)

,Component,Selected
0,EWMA correction,False
1,DST expert,False


In [25]:
# ==========================================
# REUSE STATISTICAL EVIDENCE
# ==========================================

print(
    "Tests Diebold-Mariano déjà disponibles :",
    len(
        dm_summary
    )
)


display(
    dm_summary
)

Tests Diebold-Mariano déjà disponibles : 9


,Test,Model_A,Model_B,DM_stat,p_value,mean_loss_diff
0,MSE — HAC 1 day,Patch,Event Expert,3.197891,0.001384,226.858971
1,MSE — HAC 1 day,Event Expert,Event + EWMA,0.201131,0.840596,3.219023
2,MSE — HAC 1 day,Patch,Event + EWMA,3.173882,0.001504,230.077993
3,MSE — HAC 1 week,Patch,Event Expert,3.309755,0.000934,226.858971
4,MSE — HAC 1 week,Event Expert,Event + EWMA,0.184398,0.853701,3.219023
5,MSE — HAC 1 week,Patch,Event + EWMA,3.286241,0.001015,230.077993
6,MAE — HAC 1 week,Patch,Event Expert,3.414723,0.000638,0.393709
7,MAE — HAC 1 week,Event Expert,Event + EWMA,1.147361,0.251233,0.053366
8,MAE — HAC 1 week,Patch,Event + EWMA,3.616053,0.000299,0.447075


### Complexité et validation statistique

Le Bloc 7 contient déjà des tests de Diebold-Mariano permettant de comparer
les erreurs de plusieurs variantes.

Ces résultats sont réutilisés plutôt que recalculés.

Cette approche évite de conserver une variante plus complexe uniquement
parce que son score ponctuel semble légèrement meilleur.

Une correction supplémentaire est particulièrement intéressante lorsqu'elle
apporte une amélioration :

1. mesurable sur les métriques prédictives ;
2. statistiquement défendable ;
3. suffisamment importante pour justifier sa complexité.

Le Sustainable ML rejoint donc ici la sélection de modèle : les composants
qui n'apportent pas de gain suffisamment convaincant peuvent être exclus.

In [27]:
# ==========================================
# SUSTAINABLE ML DECISION
# ==========================================

sustainable_decision = pd.DataFrame({

    "Option": [
        "Ridge only",
        "Final composite pipeline",
    ],

    "Complexity": [
        "Low",
        "Higher",
    ],

    "RMSE": [
        RIDGE_RMSE,
        FINAL_RMSE,
    ],

    "Relative_RMSE_gain_pct": [
        0.0,
        relative_rmse_gain,
    ],
})


display(
    sustainable_decision
)

,Option,Complexity,RMSE,Relative_RMSE_gain_pct
0,Ridge only,Low,209.661400,0.000000
1,Final composite pipeline,Higher,176.942561,15.605562


### Partie C — Analyse et améliorations guidées par la théorie

Le passage du Ridge au pipeline composite entraîne une augmentation de
complexité mais réduit le RMSE d'environ :

$$
15.6\%.
$$

Le pipeline final utilise plusieurs familles de modèles :

$$
Ridge + MLP + Patch\ Transformer + Event\ Expert.
$$

Le coût supplémentaire n'est donc pas nul, mais le gain prédictif est
nettement supérieur à une simple variation marginale de quelques dixièmes
de pourcent.

Par ailleurs, les variantes et corrections qui ne présentent pas de gain
suffisamment convaincant peuvent être écartées au lieu d'être empilées
systématiquement.

La sobriété du pipeline ne signifie donc pas choisir obligatoirement le
modèle le plus simple. Elle consiste à conserver uniquement la complexité
qui apporte une valeur mesurable.

Conclusion :

Le modèle final est plus complexe qu'un Ridge, mais cette complexité
s'accompagne d'une réduction du RMSE d'environ 15.6 %.

Le gain est suffisamment important pour justifier l'étude et l'utilisation
du pipeline composite.

En revanche, les composants optionnels qui n'apportent pas d'amélioration
suffisamment convaincante ne doivent pas être ajoutés systématiquement.

Le principe retenu est donc de conserver la complexité utile et d'éviter
la complexité redondante.

In [28]:
# ==========================================
# BLOCK 13 — PERFORMANCE SUMMARY
# ==========================================

block13_performance = pd.DataFrame({

    "Indicator": [
        "Ridge RMSE",
        "Final RMSE",
        "RMSE improvement (%)",
        "Final state size (MB)",
        "Candidate artifact footprint (MB)",
        "Main components",
        "Retraining performed",
        "2026 used",
    ],

    "Value": [
        RIDGE_RMSE,
        FINAL_RMSE,
        relative_rmse_gain,
        block7_size_mb,
        candidate_size_mb,
        4,
        False,
        False,
    ],
})


display(
    block13_performance
)

,Indicator,Value
0,Ridge RMSE,209.6614
1,Final RMSE,176.942561
2,RMSE improvement (%),15.605562
3,Final state size (MB),0.40396
4,Candidate artifact footprint (MB),8.624491
5,Main components,4
6,Retraining performed,False
7,2026 used,False


### Partie C — Analyse et améliorations guidées par la théorie

## Architecture Sustainable ML retenue

Le projet conserve le pipeline composite final car son amélioration
prédictive par rapport au Ridge est importante.

La stratégie de sobriété retenue repose cependant sur plusieurs principes :

- ne pas réentraîner inutilement les modèles déjà disponibles ;
- réutiliser les résultats statistiques déjà calculés ;
- ne pas conserver automatiquement toutes les variantes testées ;
- privilégier les composants apportant un gain mesurable ;
- distinguer les artefacts expérimentaux du véritable pipeline
  d'inférence ;
- éviter des estimations carbone non justifiées par des mesures
  matérielles.

Cette stratégie cherche donc à optimiser le rapport :

$$
\frac{\text{valeur prédictive}}
     {\text{complexité informatique}}.
$$

In [29]:
# ==========================================
# SAVE BLOCK 13 FINAL STATE
# ==========================================

BLOCK13_FINAL_PATH = (
    MODEL_DIR
    / "block13_final_state.joblib"
)


block13_final_state = {

    "ridge_rmse":
        RIDGE_RMSE,

    "final_rmse":
        FINAL_RMSE,

    "rmse_improvement_pct":
        relative_rmse_gain,

    "block7_state_size_mb":
        block7_size_mb,

    "candidate_artifacts_size_mb":
        candidate_size_mb,

    "main_components": [
        "Ridge",
        "MLP",
        "Patch Transformer",
        "Refined Event Expert",
    ],

    "ewma_selected":
        block7_state[
            "ewma_selected"
        ],

    "dst_expert_selected":
        block7_state[
            "dst_expert_selected"
        ],

    "retrained_models":
        False,

    "uses_2026":
        False,
}


joblib.dump(
    block13_final_state,
    BLOCK13_FINAL_PATH,
)


print(
    "Sauvegardé :",
    BLOCK13_FINAL_PATH
)

Sauvegardé : C:\Users\thiba\OneDrive\Bureau\Projet energie\models\block13_final_state.joblib


In [30]:
# ==========================================
# BLOCK 13 FINAL VALIDATION
# ==========================================

loaded_block13 = joblib.load(
    BLOCK13_FINAL_PATH
)


assert (
    loaded_block13[
        "final_rmse"
    ]
    <
    loaded_block13[
        "ridge_rmse"
    ]
)


assert (
    loaded_block13[
        "retrained_models"
    ]
    is False
)


assert (
    loaded_block13[
        "uses_2026"
    ]
    is False
)


assert (
    loaded_block13[
        "rmse_improvement_pct"
    ]
    > 0
)


print(
    "Bloc 13 : validation OK"
)

Bloc 13 : validation OK


## Conclusion du Bloc 13

Le Bloc 13 a étudié le compromis entre performance prédictive et
complexité informatique du projet.

Le Ridge constitue une solution simple avec un RMSE d'environ :

$$
209.66\ MW.
$$

Le pipeline composite final atteint environ :

$$
176.94\ MW,
$$

soit une amélioration d'environ :

$$
15.6\%.
$$

Cette amélioration est suffisamment importante pour justifier une partie
de la complexité supplémentaire du pipeline.

Le fichier d'état final du Bloc 7 ne représente qu'environ 0.404 MB.
L'inventaire plus large des artefacts associés aux modèles contient environ
8.6 MB, mais inclut également des résultats intermédiaires et des modèles
expérimentaux ; cette valeur ne représente donc pas directement la taille
du pipeline de production.

Le projet adopte finalement une stratégie de sobriété basée sur la
réutilisation des modèles existants, l'élimination des variantes inutiles
et la conservation uniquement des composants apportant un gain mesurable.

Aucun modèle n'a été réentraîné dans ce bloc et l'année 2026 reste
totalement réservée au test final.